In [11]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from pathlib import Path

In [12]:
# Create connection string
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-EKQ2K2R;"
    "DATABASE=Airbnb_DWH;"
    "Trusted_Connection=yes;"
)

# Create SQLAlchemy engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Read Bronze table
silver_df = pd.read_sql(
    "SELECT * FROM silver_airbnb",
    con=engine
)

In [3]:
silver_df

,price,room_type,room_shared,room_private,guest_capacity,is_superhost,multi,biz,cleanliness_score,guest_satisfaction_score,...,distance_to_city_center,distance_to_metro,attraction_index,normalized_attraction_index,restaurant_index,normalized_restaurant_index,longitude,latitude,city,day_type
0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,Weekdays
1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,Weekdays
2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,Weekdays
3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,...,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,Weekdays
4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,...,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,Weekdays
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51702,715.938574,Entire home/apt,False,False,6.0,False,0,1,10.0,100.0,...,0.530181,0.135447,219.402478,15.712158,438.756874,10.604584,16.37940,48.21136,Vienna,Weekends
51703,304.793960,Entire home/apt,False,False,2.0,False,0,0,8.0,86.0,...,0.810205,0.100839,204.970121,14.678608,342.182813,8.270427,16.38070,48.20296,Vienna,Weekends
51704,637.168969,Entire home/apt,False,False,2.0,False,0,0,10.0,93.0,...,0.994051,0.202539,169.073402,12.107921,282.296424,6.822996,16.38568,48.20460,Vienna,Weekends
51705,301.054157,Private room,False,True,2.0,False,0,0,10.0,87.0,...,3.044100,0.287435,109.236574,7.822803,158.563398,3.832416,16.34100,48.19200,Vienna,Weekends


In [14]:
print(silver_df.columns.tolist())

['price', 'room_type', 'room_shared', 'room_private', 'guest_capacity', 'is_superhost', 'multi', 'biz', 'cleanliness_score', 'guest_satisfaction_score', 'bedrooms', 'distance_to_city_center', 'distance_to_metro', 'attraction_index', 'normalized_attraction_index', 'restaurant_index', 'normalized_restaurant_index', 'longitude', 'latitude', 'city', 'day_type']


In [16]:
# Read Dimension Tables
dim_city = pd.read_sql("SELECT * FROM Dim_City", engine)
dim_room = pd.read_sql("SELECT * FROM Dim_Room", engine)
dim_host = pd.read_sql("SELECT * FROM Dim_Host", engine)

# Build Fact Table
fact_df = silver_df.merge(
    dim_city,
    left_on="city",
    right_on="city_name",
    how="left"
)

fact_df = fact_df.merge(
    dim_room,
    on=[
        "room_type",
        "room_shared",
        "room_private",
        "guest_capacity",
        "bedrooms"
    ],
    how="left"
)

fact_df = fact_df.merge(
    dim_host,
    on=[
        "is_superhost",
        "multi",
        "biz"
    ],
    how="left"
)

In [ ]:
print(dim_city.columns)
print(dim_room.columns)
print(dim_host.columns)

Index(['city_id', 'city_name'], dtype='object')
Index(['room_id', 'room_type', 'room_shared', 'room_private', 'guest_capacity',
       'bedrooms'],
      dtype='object')
Index(['host_id', 'is_superhost', 'multi', 'biz'], dtype='object')


In [17]:
print(fact_df.shape)

print(fact_df[["city_id", "room_id", "host_id"]].isnull().sum())

(51707, 25)
city_id    0
room_id    0
host_id    0
dtype: int64


# Feature Engineering

In [18]:
# 1. Price Category

fact_df["price_category"] = pd.qcut(
    fact_df["price"],
    q=4,
    labels=["Budget", "Standard", "Premium", "Luxury"]
)

# 2. Property Size
fact_df["property_size"] = pd.cut(
    fact_df["bedrooms"],
    bins=[-1, 1, 3, fact_df["bedrooms"].max()],
    labels=["Small", "Medium", "Large"]
)

# 3. Guest Group
fact_df["guest_group"] = pd.cut(
    fact_df["guest_capacity"],
    bins=[0, 2, 4, fact_df["guest_capacity"].max()],
    labels=["Small", "Medium", "Large"],
    include_lowest=True
)

# 4. Distance Category
fact_df["distance_category"] = pd.cut(
    fact_df["distance_to_city_center"],
    bins=[0, 2, 5, 10, fact_df["distance_to_city_center"].max()],
    labels=["City Center", "Near Center", "Suburban", "Far Away"],
    include_lowest=True
)

# 5. Premium Location
median_attraction = fact_df["normalized_attraction_index"].median()

fact_df["premium_location"] = fact_df["normalized_attraction_index"] >= median_attraction

# Convert Boolean to Business Labels
fact_df["premium_location"] = fact_df["premium_location"].replace({
    True: "Premium Area",
    False: "Regular Area"
})

# 6. Host Type
fact_df["host_type"] = fact_df["is_superhost"].replace({
    True: "Superhost",
    False: "Regular Host"
})

In [19]:
fact_df

,price,room_type,room_shared,room_private,guest_capacity,is_superhost,multi,biz,cleanliness_score,guest_satisfaction_score,...,city_id,city_name,room_id,host_id,price_category,property_size,guest_group,distance_category,premium_location,host_type
0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,...,4,Amsterdam,59,5,Standard,Small,Small,Suburban,Regular Area,Regular Host
1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,...,4,Amsterdam,42,1,Luxury,Small,Medium,City Center,Premium Area,Regular Host
2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,...,4,Amsterdam,59,4,Premium,Small,Small,Suburban,Regular Area,Regular Host
3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,...,4,Amsterdam,7,4,Luxury,Medium,Medium,City Center,Premium Area,Regular Host
4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,...,4,Amsterdam,59,6,Luxury,Small,Small,City Center,Premium Area,Superhost
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51702,715.938574,Entire home/apt,False,False,6.0,False,0,1,10.0,100.0,...,7,Vienna,45,4,Luxury,Medium,Large,City Center,Premium Area,Regular Host
51703,304.793960,Entire home/apt,False,False,2.0,False,0,0,8.0,86.0,...,7,Vienna,43,1,Premium,Small,Small,City Center,Premium Area,Regular Host
51704,637.168969,Entire home/apt,False,False,2.0,False,0,0,10.0,93.0,...,7,Vienna,43,1,Luxury,Small,Small,City Center,Premium Area,Regular Host
51705,301.054157,Private room,False,True,2.0,False,0,0,10.0,87.0,...,7,Vienna,59,1,Premium,Small,Small,Near Center,Regular Area,Regular Host


In [20]:
fact_df.isna().sum()

price                          0
room_type                      0
room_shared                    0
room_private                   0
guest_capacity                 0
is_superhost                   0
multi                          0
biz                            0
cleanliness_score              0
guest_satisfaction_score       0
bedrooms                       0
distance_to_city_center        0
distance_to_metro              0
attraction_index               0
normalized_attraction_index    0
restaurant_index               0
normalized_restaurant_index    0
longitude                      0
latitude                       0
city                           0
day_type                       0
city_id                        0
city_name                      0
room_id                        0
host_id                        0
price_category                 0
property_size                  0
guest_group                    0
distance_category              0
premium_location               0
host_type 

## Save Silver Dataset

In [21]:
output_path = Path("../datasets/processed")
output_file = output_path / "gold_airbnb.csv"

silver_df.to_csv(output_file , index= False)

## Load into SQL Server

In [22]:
fact_df.to_sql(
    name="Fact_Listings",
    con=engine,
    if_exists="replace",
    index=False
)

50